# 34 · Weight-change audit (CPU, ~40 min; resumable; no data, no evaluation)
For every bottleneck run (8 models × 5 seeds × 5 folds × 2 schedules) the initial weights are rebuilt exactly as in training
(`clear_session`, `set_seeds(seed)`, `build_p2_model`) and compared with the saved best weights, layer by layer.
Writes only `results/p2_rev5/addl/D_weight_change.csv`.

In [ ]:
# 1 · Drive, code
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import os, sys, glob, json, gc
import numpy as np, pandas as pd
PROJECT = '/content/drive/MyDrive/quantum-placement'; RES = f'{PROJECT}/results'
OUT = f'{RES}/p2_rev5/addl'; os.makedirs(OUT, exist_ok=True)
ROOTS = {'train': f'{RES}/p2_jit_runs_train', 'val': f'{RES}/p2_jit_runs_val', 'train4': f'{RES}/p2_rev4_runs_train', 'val4': f'{RES}/p2_rev4_runs_val'}
SCHED = {'train': 'train', 'val': 'val', 'train4': 'train', 'val4': 'val'}
QSEG_DIR = f'{PROJECT}/code'
if not os.path.exists(f'{QSEG_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True); assert hits, 'qseg package not found on Drive'
    QSEG_DIR = os.path.dirname(os.path.dirname(hits[0]))
for p_ in (QSEG_DIR, f'{PROJECT}/code'):
    if p_ not in sys.path: sys.path.insert(0, p_)
import tensorflow as tf, qseg2, qseg2_rev4; qseg2_rev4.patch()
from qseg.config import Config
cfg = Config(work_dir=OUT); SHAPE = (cfg.image_size, cfg.image_size, cfg.in_channels)
print('qseg package:', QSEG_DIR, '| TF', tf.__version__)

In [ ]:
# 2 · Compare initial and trained weights of the bottleneck layers (resumable)
LAYERS = ('pre_dense', 'pre_conv', 'quantum_circuit', 'classical_replacement', 'post_dense', 'post_conv', 'bottleneck_conv2', 'dec1_conv1')
def weights(name, seed=None, path=None):
    tf.keras.backend.clear_session(); tf.keras.mixed_precision.set_global_policy('float32')
    if seed is not None: qseg2.set_seeds(seed)
    m = qseg2.build_p2_model(name, SHAPE, cfg.n_qubits, cfg.n_q_layers)
    if path: m.load_weights(path)
    w = {l.name: [v.numpy().astype(np.float64) for v in l.weights if v.trainable] for l in m.layers if l.trainable_weights}
    del m; gc.collect(); return w
dst = f'{OUT}/D_weight_change.csv'
done = pd.read_csv(dst) if os.path.exists(dst) else pd.DataFrame(); rows = done.to_dict('records')
seen = set(zip(done.run, done.schedule)) if len(done) else set()
runs = [(k, d) for k, r in ROOTS.items() for d in sorted(glob.glob(f'{r}/*_fold*_seed*')) if os.path.exists(f'{d}/best.weights.h5')]
runs = [(k, d) for k, d in runs if json.load(open(f'{d}/summary.json'))['model'] not in ('full', 'unet')]
print('bottleneck runs:', len(runs))
for n, (key, d) in enumerate(runs):
    s = json.load(open(f'{d}/summary.json')); rid = os.path.basename(d)
    if (rid, SCHED[key]) in seen: continue
    w0 = weights(s['model'], int(s['seed'])); w1 = weights(s['model'], path=f'{d}/best.weights.h5')
    for ln in LAYERS:
        if ln not in w0: continue
        a = np.concatenate([x.ravel() for x in w0[ln]]); b = np.concatenate([x.ravel() for x in w1[ln]]); dlt = b - a
        if ln == 'quantum_circuit': dlt = (dlt + np.pi) % (2 * np.pi) - np.pi      # angles: shortest difference
        rows.append(dict(run=rid, schedule=SCHED[key], model=s['model'], fold=int(s['fold']), seed=int(s['seed']), layer=ln, n=a.size,
                         max_abs_change=float(np.abs(dlt).max()), rms_change=float(np.sqrt((dlt ** 2).mean())),
                         rel_change=float(np.linalg.norm(dlt) / max(np.linalg.norm(a), 1e-12)), rms_init=float(np.sqrt((a ** 2).mean()))))
    if n % 20 == 0:
        pd.DataFrame(rows).to_csv(dst, index=False); print(n, SCHED[key], rid, flush=True)
D = pd.DataFrame(rows); D.to_csv(dst, index=False)
print(D.groupby(['model', 'layer'])[['max_abs_change', 'rms_change', 'rel_change']].median().round(4).to_string())
print('DONE')